# Model Exploration — Data Overlap EDA + 3-Model Evaluation

AI-assisted (Claude Code, claude.ai) — https://claude.ai

**Part 1** — Data overlap EDA  
**Part 2** — 3-class (A/B/C) model evaluation  
**Part 3** — Binary (A vs not-A) model evaluation  
**Part 4** — Score threshold experiment: where are reviews predictive? (EX1-EX6)
- Sensitivity/specificity sweep across score thresholds
- ROC curves per threshold
- Red-flag and green-flag review words (what to look for)
- User comfort level guide

Split: 60/20/20 train/val/test, stratified.

In [ ]:
import glob, os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix, hstack
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, f1_score)
from sklearn.model_selection import GridSearchCV, train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
sns.set_palette("Set2")

RS = 42
GRADES = ["A", "B", "C"]
GMAP = {"A": 0, "B": 1, "C": 2}
BIN_LABELS = ["A (pass)", "not-A (fail)"]
ROOT = os.path.dirname(os.getcwd())
RAW = os.path.join(ROOT, "data", "raw")

# Part 1 — Data Overlap EDA
## 1.1 Load data

In [ ]:
def load_inspections(raw_dir):
    """Load and concatenate all yearly inspection files."""
    files = sorted(glob.glob(os.path.join(raw_dir, "inspections_20*.csv")))
    frames = []
    for path in files:
        year = int(os.path.basename(path).split("_")[1].split(".")[0])
        df = pd.read_csv(path); df["year"] = year; frames.append(df)
    insp = pd.concat(frames, ignore_index=True)
    insp["grade"] = insp["grade"].str.strip().str.upper()
    insp["score"] = pd.to_numeric(insp["score"], errors="coerce")
    return insp

def load_google(raw_dir):
    """Load Google reviews with numeric coercion."""
    gr = pd.read_csv(os.path.join(raw_dir, "google_reviews.csv"))
    for col in ["google_rating", "google_review_count", "match_score"]:
        gr[col] = pd.to_numeric(gr[col], errors="coerce")
    return gr

insp = load_inspections(RAW)
google = load_google(RAW)
print(f"Inspections: {len(insp):,} rows, {insp['state_id'].nunique():,} restaurants")
print(f"Google: {len(google):,} rows, {google['state_id'].nunique():,} restaurants")

## 1.2 Overlap

In [ ]:
def compute_overlap(insp_df, google_df):
    """Set overlap between inspections and Google by state_id."""
    iids = set(insp_df["state_id"].unique())
    gids = set(google_df["state_id"].unique())
    return {"insp": len(iids), "google": len(gids), "both": len(iids & gids),
            "insp_only": len(iids - gids), "google_only": len(gids - iids), "ids": iids & gids}

ov = compute_overlap(insp, google)
for k in ["insp", "google", "both", "insp_only", "google_only"]:
    print(f"  {k:12s}: {ov[k]:,}")
print(f"  match rate:  {100*ov['both']/ov['insp']:.1f}%")

fig, ax = plt.subplots(figsize=(8, 2.5))
left = 0
for lbl, sz, c in [("Insp only", ov["insp_only"], "#66c2a5"), ("Both", ov["both"], "#8da0cb"), ("Google only", ov["google_only"], "#fc8d62")]:
    ax.barh(0, sz, left=left, color=c, edgecolor="white", height=0.6, label=f"{lbl}: {sz:,}")
    ax.text(left + sz/2, 0, f"{sz:,}", ha="center", va="center", fontweight="bold")
    left += sz
ax.set_yticks([]); ax.legend(loc="upper right"); ax.set_title("Data Overlap")
plt.tight_layout(); plt.show()

## 1.3 Grade distribution — matched vs unmatched

In [ ]:
def grade_matched_vs_unmatched(insp_df, overlap_ids):
    """Side-by-side grade bars for matched vs unmatched restaurants."""
    valid = insp_df[insp_df["grade"].isin(GRADES)].copy()
    valid["has_google"] = valid["state_id"].isin(overlap_ids)
    per_rest = valid.groupby("state_id").agg(grade=("grade","last"), has_google=("has_google","first")).reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (label, sub) in zip(axes, per_rest.groupby("has_google")):
        counts = sub["grade"].value_counts().reindex(GRADES, fill_value=0)
        pcts = counts / counts.sum() * 100
        bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", 3))
        for b, p in zip(bars, pcts):
            ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{p:.1f}%", ha="center", va="bottom", fontsize=9)
        ax.set_title(f"{'With' if label else 'No'} Google data (n={len(sub):,})")
        ax.set_xlabel("Grade"); ax.set_ylabel("Count")
    plt.suptitle("Grade: matched vs unmatched", y=1.02); plt.tight_layout(); plt.show()

grade_matched_vs_unmatched(insp, ov["ids"])

## 1.4 Match quality

In [ ]:
def plot_match_quality(gdf):
    """Match score distribution and coverage tradeoff."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(gdf["match_score"].dropna(), bins=50, edgecolor="white", linewidth=0.4)
    for t in [30, 50, 80]:
        axes[0].axvline(t, color="red", ls="--", alpha=0.6)
    axes[0].set_xlabel("Match score"); axes[0].set_title("Match score distribution")
    thresholds = range(0, 101, 5)
    retained = [gdf[gdf["match_score"]>=t]["state_id"].nunique() for t in thresholds]
    axes[1].plot(list(thresholds), retained, "o-", ms=4)
    axes[1].set_xlabel("Min threshold"); axes[1].set_ylabel("Restaurants"); axes[1].set_title("Coverage vs quality")
    plt.tight_layout(); plt.show()
    for t in [0, 20, 50, 80]:
        n = gdf[gdf["match_score"]>=t]["state_id"].nunique()
        nt = gdf[(gdf["match_score"]>=t) & gdf["google_reviews"].notna()]["state_id"].nunique()
        print(f"  >={t:3d}: {n:,} restaurants, {nt:,} with text")

plot_match_quality(google)

## 1.5 Review text stats

In [ ]:
def plot_review_stats(gdf):
    """Text length, review count, and rating vs length."""
    ht = gdf[gdf["google_reviews"].notna()].copy()
    ht["tlen"] = ht["google_reviews"].str.len()
    ht["nrev"] = ht["google_reviews"].str.count(r"\|\|\|") + 1
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(ht["tlen"], bins=50, edgecolor="white", lw=0.3); axes[0].set_xlabel("Chars"); axes[0].set_title(f"Text length (n={len(ht):,})")
    axes[1].hist(ht["nrev"], bins=range(1, ht["nrev"].max()+2), edgecolor="white", lw=0.3, color="#fc8d62"); axes[1].set_xlabel("Reviews"); axes[1].set_title("Reviews per restaurant")
    axes[2].scatter(ht["google_rating"], ht["tlen"], alpha=0.15, s=8); axes[2].set_xlabel("Rating"); axes[2].set_ylabel("Text len"); axes[2].set_title("Rating vs text length")
    plt.tight_layout(); plt.show()
    print(f"Reviews/rest: mean={ht['nrev'].mean():.1f}, median={ht['nrev'].median():.0f}")
    print(f"Text len: mean={ht['tlen'].mean():.0f}, median={ht['tlen'].median():.0f}")

plot_review_stats(google)

## 1.6 Google rating vs grade

In [ ]:
def plot_rating_grade(idf, gdf):
    """Box+strip of Google rating by inspection grade."""
    latest = idf[idf["grade"].isin(GRADES)].groupby("state_id")["grade"].last().reset_index()
    gd = gdf.dropna(subset=["google_rating"]).drop_duplicates("state_id")
    m = latest.merge(gd[["state_id","google_rating"]], on="state_id")
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=m, x="grade", y="google_rating", order=GRADES, ax=ax, showfliers=False)
    sns.stripplot(data=m, x="grade", y="google_rating", order=GRADES, ax=ax, alpha=0.15, size=2, jitter=True, color="black")
    ax.set_title("Google rating by inspection grade")
    plt.tight_layout(); plt.show()
    print(m.groupby("grade")["google_rating"].describe().round(3))

plot_rating_grade(insp, google)

---
# Part 2 — Modeling Setup
## 2.1 Build dataset & split

In [ ]:
def build_model_df(raw_dir):
    """Per-restaurant frame with latest grade + Google features."""
    idf = load_inspections(raw_dir)
    gdf = load_google(raw_dir)
    gg = gdf[(gdf["match_score"]>=20) & gdf["google_reviews"].notna() & gdf["google_rating"].notna()]
    gg = gg.sort_values("match_score", ascending=False).drop_duplicates("state_id")
    valid = idf[idf["grade"].isin(GRADES)].copy()
    rest = valid.groupby("state_id").agg(
        grade=("grade","last"), num_inspections=("score","count"),
        establishment_type=("establishment_type","first"), county_code=("county_code","first"),
    ).reset_index()
    df = rest.merge(gg[["state_id","google_rating","google_review_count","google_reviews","match_score"]], on="state_id", how="inner")
    df["grade_enc"] = df["grade"].map(GMAP)
    df["bin_target"] = (df["grade_enc"] > 0).astype(int)
    df["review_count_log"] = np.log1p(df["google_review_count"].fillna(0))
    df["est_type"] = pd.to_numeric(df["establishment_type"].str.split(" - ").str[0], errors="coerce")
    print(f"Dataset: {len(df):,} restaurants")
    print(f"3-class: {dict(df['grade'].value_counts())}")
    print(f"Binary:  A={int((df['bin_target']==0).sum())}, not-A={int(df['bin_target'].sum())}")
    return df

mdf = build_model_df(RAW)

In [ ]:
def split3(df, target, train_frac=0.6):
    """Stratified 60/20/20 split."""
    tr, tmp = train_test_split(df, test_size=1-train_frac, random_state=RS, stratify=df[target])
    va, te = train_test_split(tmp, test_size=0.5, random_state=RS, stratify=tmp[target])
    for n, s in [("Train",tr),("Val",va),("Test",te)]:
        print(f"{n:5s}: {len(s):,} | {dict(s[target].value_counts().sort_index())}")
    return tr, va, te

train_df, val_df, test_df = split3(mdf, "grade_enc")

## 2.2 Features

In [ ]:
SFEAT = ["google_rating", "review_count_log", "num_inspections", "county_code", "est_type"]

def get_Xy(df, cols, target):
    """Extract feature matrix and target."""
    return df[cols].fillna(0).values, df[target].values

X_tr_s, y_tr = get_Xy(train_df, SFEAT, "grade_enc")
X_va_s, y_va = get_Xy(val_df, SFEAT, "grade_enc")
X_te_s, y_te = get_Xy(test_df, SFEAT, "grade_enc")

# Binary targets
y_tr_b = train_df["bin_target"].values
y_va_b = val_df["bin_target"].values
y_te_b = test_df["bin_target"].values

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1,2), min_df=3, max_df=0.95, sublinear_tf=True)
tr_txt = train_df["google_reviews"].fillna("").tolist()
va_txt = val_df["google_reviews"].fillna("").tolist()
te_txt = test_df["google_reviews"].fillna("").tolist()
X_tr_tf = tfidf.fit_transform(tr_txt)
X_va_tf = tfidf.transform(va_txt)
X_te_tf = tfidf.transform(te_txt)

# Combined
X_tr_c = hstack([csr_matrix(X_tr_s), X_tr_tf])
X_va_c = hstack([csr_matrix(X_va_s), X_va_tf])
X_te_c = hstack([csr_matrix(X_te_s), X_te_tf])

print(f"Structured: {X_tr_s.shape}, TF-IDF: {X_tr_tf.shape}, Combined: {X_tr_c.shape}")
print(f"Binary: train not-A={y_tr_b.sum()}, val={y_va_b.sum()}, test={y_te_b.sum()}")

## 2.3 Evaluation helper

In [ ]:
def ev(model, X, yt, name, split, labels):
    """Classification report + confusion matrix."""
    yp = model.predict(X)
    mf1 = f1_score(yt, yp, average="macro", zero_division=0)
    print(f"\n--- {name} ({split}) F1={mf1:.4f} ---")
    print(classification_report(yt, yp, target_names=labels, zero_division=0))
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ConfusionMatrixDisplay.from_predictions(yt, yp, display_labels=labels, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} ({split}) F1={mf1:.3f}")
    plt.tight_layout(); plt.show()
    return {"name": name, "f1": mf1}

def train_rf(X, y, name="RF"):
    """GridSearchCV Random Forest."""
    search = GridSearchCV(
        RandomForestClassifier(random_state=RS, n_jobs=-1),
        {"n_estimators": [100,200], "max_depth": [None,10,20], "min_samples_split": [2,5], "class_weight": ["balanced",None]},
        cv=5, scoring="f1_macro", n_jobs=-1, verbose=0,
    )
    search.fit(X, y)
    print(f"{name}: {search.best_params_} (CV F1={search.best_score_:.4f})")
    return search.best_estimator_

---
# Part 2a — 3-Class Models (A/B/C)

Expected: all models predict A only due to ~2 C and ~69 B train samples.

In [ ]:
bl3 = DummyClassifier(strategy="most_frequent", random_state=RS).fit(X_tr_s, y_tr)
r_bl3 = ev(bl3, X_va_s, y_va, "Naive Baseline", "Val", GRADES)

rf3s = train_rf(X_tr_s, y_tr, "RF-struct-3c")
r_rf3s = ev(rf3s, X_va_s, y_va, "RF (structured)", "Val", GRADES)

rf3c = train_rf(X_tr_c, y_tr, "RF-combo-3c")
r_rf3c = ev(rf3c, X_va_c, y_va, "RF (struct+tfidf)", "Val", GRADES)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

class RevDS(Dataset):
    """Tokenized reviews for DistilBERT."""
    def __init__(self, enc, lab):
        self.enc, self.lab = enc, lab
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.lab[i], dtype=torch.long)
        return item
    def __len__(self): return len(self.lab)

def train_bert(tr_t, y_tr, va_t, y_va, nl, epochs=3, bs=16, lr=2e-5):
    """Fine-tune DistilBERT with class-weighted loss."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {dev}")
    tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    tr_enc = tok(tr_t, truncation=True, padding=True, max_length=256)
    va_enc = tok(va_t, truncation=True, padding=True, max_length=256)
    tr_dl = DataLoader(RevDS(tr_enc, y_tr), batch_size=bs, shuffle=True)
    va_dl = DataLoader(RevDS(va_enc, y_va), batch_size=bs)
    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=nl).to(dev)
    counts = np.bincount(y_tr, minlength=nl).astype(float)
    counts = np.where(counts==0, 1.0, counts)
    wts = torch.tensor(len(y_tr)/(nl*counts), dtype=torch.float).to(dev)
    print(f"Weights: {wts.cpu().numpy()}")
    loss_fn = torch.nn.CrossEntropyLoss(weight=wts)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for ep in range(epochs):
        model.train(); tloss = 0
        for b in tr_dl:
            opt.zero_grad()
            out = model(input_ids=b["input_ids"].to(dev), attention_mask=b["attention_mask"].to(dev))
            l = loss_fn(out.logits, b["labels"].to(dev)); l.backward(); opt.step(); tloss += l.item()
        model.eval(); preds, labs, vloss = [], [], 0
        with torch.no_grad():
            for b in va_dl:
                out = model(input_ids=b["input_ids"].to(dev), attention_mask=b["attention_mask"].to(dev))
                vloss += loss_fn(out.logits, b["labels"].to(dev)).item()
                preds.extend(out.logits.argmax(-1).cpu().numpy()); labs.extend(b["labels"].numpy())
        vf1 = f1_score(labs, preds, average="macro", zero_division=0)
        hist.append({"ep": ep+1, "tl": tloss/len(tr_dl), "vl": vloss/len(va_dl), "vf1": vf1})
        print(f"  Ep {ep+1}/{epochs} tl={hist[-1]['tl']:.4f} vl={hist[-1]['vl']:.4f} vF1={vf1:.4f}")
    return model, tok, hist, np.array(preds), np.array(labs)

def predict_bert(model, tok, texts, bs=16):
    """Batch inference."""
    dev = next(model.parameters()).device; model.eval(); ps = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], truncation=True, padding=True, max_length=256, return_tensors="pt")
        enc = {k: v.to(dev) for k, v in enc.items()}
        with torch.no_grad(): ps.extend(model(**enc).logits.argmax(-1).cpu().numpy())
    return np.array(ps)

In [ ]:
print("DistilBERT (3-class)...")
bert3, tok3, h3, vp3, vl3 = train_bert(tr_txt, y_tr.tolist(), va_txt, y_va.tolist(), nl=3)

hd = pd.DataFrame(h3)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(hd["ep"], hd["tl"], "o-", label="Train"); axes[0].plot(hd["ep"], hd["vl"], "s-", label="Val")
axes[0].legend(); axes[0].set_title("Loss (3-class)")
axes[1].plot(hd["ep"], hd["vf1"], "o-", color="green"); axes[1].set_title("Val F1 (3-class)")
plt.tight_layout(); plt.show()

bert3_f1 = f1_score(vl3, vp3, average="macro", zero_division=0)
print(f"\nDistilBERT 3-class val F1: {bert3_f1:.4f}")
print(classification_report(vl3, vp3, target_names=GRADES, zero_division=0))

In [ ]:
# 3-class comparison
r3 = pd.DataFrame([
    {"Model": "Naive Baseline", "F1": r_bl3["f1"]},
    {"Model": "RF (structured)", "F1": r_rf3s["f1"]},
    {"Model": "RF (struct+tfidf)", "F1": r_rf3c["f1"]},
    {"Model": "DistilBERT", "F1": bert3_f1},
]).sort_values("F1", ascending=False)
print("3-CLASS VAL COMPARISON")
print(r3.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(r3["Model"], r3["F1"], color=sns.color_palette("Set2", 4))
for i, v in enumerate(r3["F1"]): ax.text(v+0.005, i, f"{v:.3f}", va="center")
ax.set_xlabel("Macro F1"); ax.set_title("3-Class Val Comparison")
plt.tight_layout(); plt.show()

---
# Part 3 — Binary Models (A vs not-A)

Merging B+C gives ~71 train failures. Still imbalanced but learnable.

In [ ]:
bl2 = DummyClassifier(strategy="most_frequent", random_state=RS).fit(X_tr_s, y_tr_b)
r_bl2 = ev(bl2, X_va_s, y_va_b, "Naive Baseline", "Val", BIN_LABELS)

print("RF (structured) binary...")
rf2s = train_rf(X_tr_s, y_tr_b, "RF-struct-bin")
r_rf2s = ev(rf2s, X_va_s, y_va_b, "RF (structured)", "Val", BIN_LABELS)

print("\nRF (struct+tfidf) binary...")
rf2c = train_rf(X_tr_c, y_tr_b, "RF-combo-bin")
r_rf2c = ev(rf2c, X_va_c, y_va_b, "RF (struct+tfidf)", "Val", BIN_LABELS)

In [ ]:
print("DistilBERT (binary)...")
bert2, tok2, h2, vp2, vl2 = train_bert(tr_txt, y_tr_b.tolist(), va_txt, y_va_b.tolist(), nl=2)

hd2 = pd.DataFrame(h2)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(hd2["ep"], hd2["tl"], "o-", label="Train"); axes[0].plot(hd2["ep"], hd2["vl"], "s-", label="Val")
axes[0].legend(); axes[0].set_title("Loss (binary)")
axes[1].plot(hd2["ep"], hd2["vf1"], "o-", color="green"); axes[1].set_title("Val F1 (binary)")
plt.tight_layout(); plt.show()

bert2_f1 = f1_score(vl2, vp2, average="macro", zero_division=0)
print(f"DistilBERT binary val F1: {bert2_f1:.4f}")
print(classification_report(vl2, vp2, target_names=BIN_LABELS, zero_division=0))

In [ ]:
# Binary val comparison
rb = pd.DataFrame([
    {"Model": "Naive Baseline", "F1": r_bl2["f1"]},
    {"Model": "RF (structured)", "F1": r_rf2s["f1"]},
    {"Model": "RF (struct+tfidf)", "F1": r_rf2c["f1"]},
    {"Model": "DistilBERT", "F1": bert2_f1},
]).sort_values("F1", ascending=False)
print("BINARY VAL COMPARISON")
print(rb.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(rb["Model"], rb["F1"], color=sns.color_palette("Set2", 4))
for i, v in enumerate(rb["F1"]): ax.text(v+0.005, i, f"{v:.3f}", va="center")
ax.set_xlabel("Macro F1"); ax.set_title("Binary Val Comparison (A vs not-A)")
plt.tight_layout(); plt.show()

---
## Final Test Evaluation

Touched once — binary framing only (3-class is intractable).

In [ ]:
print("=" * 60)
print("FINAL TEST — BINARY (A vs not-A)")
print("=" * 60)

t_bl = ev(bl2, X_te_s, y_te_b, "Naive Baseline", "Test", BIN_LABELS)
t_rfs = ev(rf2s, X_te_s, y_te_b, "RF (structured)", "Test", BIN_LABELS)
t_rfc = ev(rf2c, X_te_c, y_te_b, "RF (struct+tfidf)", "Test", BIN_LABELS)

bert2_tp = predict_bert(bert2, tok2, te_txt)
bert2_tf1 = f1_score(y_te_b, bert2_tp, average="macro", zero_division=0)
print(f"\n--- DistilBERT (Test) F1={bert2_tf1:.4f} ---")
print(classification_report(y_te_b, bert2_tp, target_names=BIN_LABELS, zero_division=0))
fig, ax = plt.subplots(figsize=(4, 3.5))
ConfusionMatrixDisplay.from_predictions(y_te_b, bert2_tp, display_labels=BIN_LABELS, ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"DistilBERT (Test) F1={bert2_tf1:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
final = pd.DataFrame([
    {"Model": "Naive Baseline", "Test F1": t_bl["f1"]},
    {"Model": "RF (structured)", "Test F1": t_rfs["f1"]},
    {"Model": "RF (struct+tfidf)", "Test F1": t_rfc["f1"]},
    {"Model": "DistilBERT", "Test F1": bert2_tf1},
]).sort_values("Test F1", ascending=False)

print("\nFINAL TEST COMPARISON (BINARY)")
print(final.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(final["Model"], final["Test F1"], color=sns.color_palette("Set2", 4))
for i, v in enumerate(final["Test F1"]): ax.text(v+0.005, i, f"{v:.3f}", va="center")
ax.set_xlabel("Macro F1"); ax.set_title("Final Test — Binary (A vs not-A)")
plt.tight_layout(); plt.show()

---
## Key Findings

| Question | Finding |
|----------|--------|
| 3-class feasibility | Intractable — C has ~2 train samples |
| Binary baseline | ~0.50 macro-F1 (predicts pass only) |
| RF structured vs baseline | Check if balanced weighting finds B patterns |
| TF-IDF helps RF? | Review text may flag hygiene/cleanliness |
| DistilBERT vs RF | BERT should capture nuanced food-safety language |
| Dominant challenge | Class imbalance — ~98.7% A even in binary |

---
# Part 4 — Score Threshold Experiment: Where Are Reviews Predictive?

**Motivation (EX1, EX5):** The A/B/C grade cutoffs are arbitrary (90, 80) and
produce extreme imbalance. Instead, we ask: **at what inspection score threshold
do Google reviews become predictive?**

We sweep thresholds from 90 to 98 and train a model at each, measuring
sensitivity (recall on the "at-risk" class) and specificity (recall on the
"safe" class). This tells us:

1. Where the signal lives in review text
2. What threshold a user should choose based on their comfort level
3. What words in reviews predict lower hygiene scores

**Decision this informs (EX5):** Whether to deploy the model as a binary
A-vs-not-A classifier or as a configurable threshold tool, and which threshold
maximizes the practical F1 tradeoff.

## 4.1 Score threshold landscape

How does class balance change as we move the "at-risk" cutoff?

In [ ]:
# We need the actual score in our modeling df. Rebuild with score included.
def build_score_df(raw_dir):
    """Per-restaurant frame including last inspection score for threshold experiments."""
    idf = load_inspections(raw_dir)
    gdf = load_google(raw_dir)
    gg = gdf[(gdf["match_score"]>=20) & gdf["google_reviews"].notna() & gdf["google_rating"].notna()]
    gg = gg.sort_values("match_score", ascending=False).drop_duplicates("state_id")
    valid = idf[idf["grade"].isin(GRADES) & idf["score"].notna()].copy()
    rest = valid.groupby("state_id").agg(
        last_score=("score", "last"), mean_score=("score", "mean"),
        min_score=("score", "min"), num_inspections=("score", "count"),
        county_code=("county_code", "first"),
        establishment_type=("establishment_type", "first"),
    ).reset_index()
    df = rest.merge(gg[["state_id","google_rating","google_review_count","google_reviews","match_score"]], on="state_id", how="inner")
    df["review_count_log"] = np.log1p(df["google_review_count"].fillna(0))
    df["est_type"] = pd.to_numeric(df["establishment_type"].str.split(" - ").str[0], errors="coerce")
    return df

sdf = build_score_df(RAW)
print(f"Score dataset: {len(sdf):,} restaurants")
print(f"Score stats: mean={sdf['last_score'].mean():.1f}, median={sdf['last_score'].median():.1f}")

# Threshold landscape
thresholds = [90, 91, 92, 93, 94, 95, 96, 97, 98]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = [(sdf["last_score"] < t).sum() for t in thresholds]
pcts = [100 * c / len(sdf) for c in counts]

axes[0].bar(thresholds, counts, color="#fc8d62", edgecolor="white")
for t, c, p in zip(thresholds, counts, pcts):
    axes[0].text(t, c, f"{c:,}\n({p:.0f}%)", ha="center", va="bottom", fontsize=8)
axes[0].set_xlabel("Score threshold (\"at-risk\" if score < threshold)")
axes[0].set_ylabel("Number of at-risk restaurants")
axes[0].set_title("At-risk class size by threshold")

axes[1].plot(thresholds, pcts, "o-", color="#8da0cb", linewidth=2, markersize=8)
axes[1].axhline(20, color="green", ls="--", alpha=0.5, label="20% minority (workable)")
axes[1].axhline(5, color="red", ls="--", alpha=0.5, label="5% minority (hard)")
axes[1].set_xlabel("Score threshold")
axes[1].set_ylabel("% at-risk")
axes[1].set_title("Minority class % by threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nThreshold summary:")
for t, c, p in zip(thresholds, counts, pcts):
    print(f"  < {t}: {c:,} at-risk ({p:.1f}%), {len(sdf)-c:,} safe ({100-p:.1f}%)")

## 4.2 Threshold sweep — RF + TF-IDF sensitivity/specificity

Train RF (struct+tfidf) at each threshold. Track sensitivity (recall on at-risk),
specificity (recall on safe), and macro-F1.

In [ ]:
from sklearn.metrics import recall_score, precision_score, roc_auc_score, roc_curve

def threshold_sweep(sdf, thresholds, sfeat_cols):
    """Train RF at each score threshold, collect sensitivity/specificity metrics."""
    results = []
    tfidf_sweep = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1,2),
                                   min_df=3, max_df=0.95, sublinear_tf=True)

    for thresh in thresholds:
        # Create binary target at this threshold
        sdf_t = sdf.copy()
        sdf_t["target"] = (sdf_t["last_score"] < thresh).astype(int)
        n_pos = sdf_t["target"].sum()

        if n_pos < 10:
            print(f"  < {thresh}: only {n_pos} positives, skipping")
            continue

        # Split (stratified)
        tr, tmp = train_test_split(sdf_t, test_size=0.4, random_state=RS, stratify=sdf_t["target"])
        va, te = train_test_split(tmp, test_size=0.5, random_state=RS, stratify=tmp["target"])

        # Features
        X_tr_struct = tr[sfeat_cols].fillna(0).values
        X_va_struct = va[sfeat_cols].fillna(0).values
        X_te_struct = te[sfeat_cols].fillna(0).values

        X_tr_tfidf = tfidf_sweep.fit_transform(tr["google_reviews"].fillna(""))
        X_va_tfidf = tfidf_sweep.transform(va["google_reviews"].fillna(""))
        X_te_tfidf = tfidf_sweep.transform(te["google_reviews"].fillna(""))

        X_tr_all = hstack([csr_matrix(X_tr_struct), X_tr_tfidf])
        X_va_all = hstack([csr_matrix(X_va_struct), X_va_tfidf])

        y_tr_t = tr["target"].values
        y_va_t = va["target"].values

        # Train with balanced class weights
        rf = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight="balanced",
                                     random_state=RS, n_jobs=-1)
        rf.fit(X_tr_all, y_tr_t)

        y_pred = rf.predict(X_va_all)
        y_proba = rf.predict_proba(X_va_all)

        sensitivity = recall_score(y_va_t, y_pred, pos_label=1, zero_division=0)
        specificity = recall_score(y_va_t, y_pred, pos_label=0, zero_division=0)
        precision = precision_score(y_va_t, y_pred, pos_label=1, zero_division=0)
        mf1 = f1_score(y_va_t, y_pred, average="macro", zero_division=0)

        # AUC if both classes present in predictions
        try:
            auc = roc_auc_score(y_va_t, y_proba[:, 1])
        except ValueError:
            auc = np.nan

        n_train_pos = y_tr_t.sum()
        n_val_pos = y_va_t.sum()

        results.append({
            "threshold": thresh,
            "train_pos": int(n_train_pos), "val_pos": int(n_val_pos),
            "minority_pct": 100 * n_pos / len(sdf_t),
            "sensitivity": sensitivity, "specificity": specificity,
            "precision": precision, "macro_f1": mf1, "auc": auc,
            "model": rf, "tfidf": tfidf_sweep,
            "X_va": X_va_all, "y_va": y_va_t, "y_proba": y_proba,
        })
        print(f"  < {thresh}: pos={n_train_pos}/{n_val_pos} (tr/va) | "
              f"sens={sensitivity:.3f} spec={specificity:.3f} F1={mf1:.3f} AUC={auc:.3f}")

    return results

SFEAT_SWEEP = ["google_rating", "review_count_log", "num_inspections", "county_code", "est_type"]
sweep = threshold_sweep(sdf, thresholds, SFEAT_SWEEP)

## 4.3 Sensitivity vs Specificity across thresholds

This is the key plot for user-facing threshold selection.

In [ ]:
def plot_threshold_sweep(sweep_results):
    """Sensitivity/specificity/F1/AUC across score thresholds."""
    rdf = pd.DataFrame([{k: v for k, v in r.items() if k not in ("model","tfidf","X_va","y_va","y_proba")}
                         for r in sweep_results])

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Sensitivity vs Specificity
    axes[0].plot(rdf["threshold"], rdf["sensitivity"], "o-", label="Sensitivity (recall at-risk)", color="#e41a1c", lw=2)
    axes[0].plot(rdf["threshold"], rdf["specificity"], "s-", label="Specificity (recall safe)", color="#377eb8", lw=2)
    axes[0].fill_between(rdf["threshold"], rdf["sensitivity"], rdf["specificity"], alpha=0.1, color="gray")
    axes[0].set_xlabel("Score threshold (at-risk if < this)")
    axes[0].set_ylabel("Rate")
    axes[0].set_title("Sensitivity vs Specificity")
    axes[0].legend()
    axes[0].set_ylim(0, 1.05)
    # Annotate the crossover or best tradeoff
    best_f1_idx = rdf["macro_f1"].idxmax()
    best_t = rdf.loc[best_f1_idx, "threshold"]
    axes[0].axvline(best_t, color="green", ls=":", alpha=0.7)
    axes[0].text(best_t + 0.2, 0.5, f"Best F1\n(< {best_t})", fontsize=9, color="green")

    # Macro F1 and AUC
    axes[1].plot(rdf["threshold"], rdf["macro_f1"], "o-", label="Macro F1", color="#4daf4a", lw=2)
    axes[1].plot(rdf["threshold"], rdf["auc"], "D-", label="AUC", color="#984ea3", lw=2)
    axes[1].set_xlabel("Score threshold")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Model quality by threshold")
    axes[1].legend()
    axes[1].set_ylim(0, 1.05)

    # Precision vs minority %
    axes[2].plot(rdf["threshold"], rdf["precision"], "o-", label="Precision (at-risk)", color="#ff7f00", lw=2)
    ax2r = axes[2].twinx()
    ax2r.plot(rdf["threshold"], rdf["minority_pct"], "s--", label="% at-risk in data", color="gray", lw=1.5)
    ax2r.set_ylabel("% at-risk in data", color="gray")
    axes[2].set_xlabel("Score threshold")
    axes[2].set_ylabel("Precision")
    axes[2].set_title("Precision vs class balance")
    axes[2].legend(loc="upper left")
    ax2r.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

    # Summary table
    print("\nThreshold sweep results:")
    print(rdf[["threshold", "minority_pct", "sensitivity", "specificity", "precision", "macro_f1", "auc"]].to_string(index=False, float_format="%.3f"))

    return rdf

sweep_df = plot_threshold_sweep(sweep)

## 4.4 ROC curves at key thresholds

ROC curves show the full sensitivity/specificity tradeoff at each threshold,
not just the default decision boundary.

In [ ]:
def plot_roc_curves(sweep_results, key_thresholds=None):
    """Overlay ROC curves for selected thresholds."""
    if key_thresholds is None:
        key_thresholds = [r["threshold"] for r in sweep_results]

    fig, ax = plt.subplots(figsize=(7, 6))
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(key_thresholds)))

    for r, color in zip(sweep_results, colors):
        if r["threshold"] not in key_thresholds:
            continue
        fpr, tpr, _ = roc_curve(r["y_va"], r["y_proba"][:, 1])
        ax.plot(fpr, tpr, lw=2, color=color, label=f"< {r['threshold']} (AUC={r['auc']:.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.3)
    ax.set_xlabel("False Positive Rate (1 - Specificity)")
    ax.set_ylabel("True Positive Rate (Sensitivity)")
    ax.set_title("ROC Curves by Score Threshold")
    ax.legend(title="At-risk if score...", loc="lower right")
    plt.tight_layout()
    plt.show()

plot_roc_curves(sweep, key_thresholds=[90, 93, 95, 97])

## 4.5 What to look for in reviews — TF-IDF feature importance

At the best-performing threshold, which words/phrases in Google reviews are
most predictive of a lower inspection score? These are the signals a user
should watch for when reading reviews.

In [ ]:
def extract_review_signals(sweep_results, sfeat_cols, top_n=30):
    """Extract and visualize the most predictive review words at the best threshold.
    
    Splits feature importances into 'red flags' (predict at-risk) and
    'green flags' (predict safe) based on which class they push toward.
    """
    # Find best threshold by macro F1
    best = max(sweep_results, key=lambda r: r["macro_f1"])
    thresh = best["threshold"]
    rf_model = best["model"]
    tfidf_model = best["tfidf"]

    print(f"Best threshold: < {thresh} (F1={best['macro_f1']:.3f}, AUC={best['auc']:.3f})")
    print(f"Sensitivity={best['sensitivity']:.3f}, Specificity={best['specificity']:.3f}\n")

    # Feature names: structured features first, then TF-IDF terms
    feature_names = list(sfeat_cols) + list(tfidf_model.get_feature_names_out())
    importances = rf_model.feature_importances_

    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)

    # Separate structured vs text features
    struct_imp = imp_df[imp_df["feature"].isin(sfeat_cols)].head(10)
    text_imp = imp_df[~imp_df["feature"].isin(sfeat_cols)].head(top_n)

    # --- Directional analysis: which words predict at-risk vs safe? ---
    # Use the model's prediction on val set to determine directionality
    X_va = best["X_va"]
    y_va = best["y_va"]
    n_struct = len(sfeat_cols)

    # For text features, compare mean TF-IDF weight in at-risk vs safe restaurants
    if hasattr(X_va, "toarray"):
        X_dense = X_va.toarray()
    else:
        X_dense = X_va

    text_features = X_dense[:, n_struct:]  # just the TF-IDF columns
    tfidf_names = list(tfidf_model.get_feature_names_out())

    at_risk_mask = y_va == 1
    safe_mask = y_va == 0

    mean_atrisk = text_features[at_risk_mask].mean(axis=0) if at_risk_mask.sum() > 0 else np.zeros(len(tfidf_names))
    mean_safe = text_features[safe_mask].mean(axis=0) if safe_mask.sum() > 0 else np.zeros(len(tfidf_names))

    # Direction = mean_atrisk - mean_safe (positive = more common in at-risk reviews)
    direction = mean_atrisk - mean_safe

    dir_df = pd.DataFrame({
        "word": tfidf_names,
        "importance": importances[n_struct:],
        "direction": direction,
        "mean_atrisk": mean_atrisk,
        "mean_safe": mean_safe,
    })
    dir_df["signed_importance"] = dir_df["importance"] * np.sign(dir_df["direction"])

    # Top red flags (predict at-risk — higher TF-IDF in failing restaurants)
    red_flags = dir_df[dir_df["direction"] > 0].nlargest(top_n, "importance")
    # Top green flags (predict safe — higher TF-IDF in passing restaurants)
    green_flags = dir_df[dir_df["direction"] < 0].nlargest(top_n, "importance")

    # --- Plots ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))

    # Structured feature importance
    axes[0].barh(struct_imp["feature"][::-1], struct_imp["importance"][::-1], color="#8da0cb")
    axes[0].set_xlabel("Feature importance")
    axes[0].set_title(f"Structured features (threshold < {thresh})")

    # Red flags
    rf_plot = red_flags.head(20)
    axes[1].barh(rf_plot["word"][::-1], rf_plot["importance"][::-1], color="#e41a1c")
    axes[1].set_xlabel("Feature importance")
    axes[1].set_title(f"RED FLAGS — words predicting at-risk (score < {thresh})")

    # Green flags
    gf_plot = green_flags.head(20)
    axes[2].barh(gf_plot["word"][::-1], gf_plot["importance"][::-1], color="#4daf4a")
    axes[2].set_xlabel("Feature importance")
    axes[2].set_title(f"GREEN FLAGS — words predicting safe (score >= {thresh})")

    plt.tight_layout()
    plt.show()

    # Print summaries
    print(f"\n{'='*60}")
    print(f"RED FLAGS — review words associated with LOWER inspection scores (< {thresh})")
    print(f"{'='*60}")
    for _, row in red_flags.head(15).iterrows():
        print(f"  {row['word']:30s}  imp={row['importance']:.4f}  "
              f"avg_atrisk={row['mean_atrisk']:.4f}  avg_safe={row['mean_safe']:.4f}")

    print(f"\n{'='*60}")
    print(f"GREEN FLAGS — review words associated with HIGHER inspection scores (>= {thresh})")
    print(f"{'='*60}")
    for _, row in green_flags.head(15).iterrows():
        print(f"  {row['word']:30s}  imp={row['importance']:.4f}  "
              f"avg_atrisk={row['mean_atrisk']:.4f}  avg_safe={row['mean_safe']:.4f}")

    return red_flags, green_flags, dir_df

red_flags, green_flags, directional_df = extract_review_signals(sweep, SFEAT_SWEEP)

## 4.6 Red flags at multiple thresholds — what changes?

Do the same words predict failure at strict (< 98) vs lenient (< 92) thresholds?
If the words change, that tells us different review signals matter at different
hygiene standards.

In [ ]:
def compare_red_flags_across_thresholds(sweep_results, sfeat_cols, key_thresholds=None, top_n=15):
    """Show how the top red-flag words change across thresholds."""
    if key_thresholds is None:
        key_thresholds = [92, 95, 97]

    threshold_words = {}
    for r in sweep_results:
        if r["threshold"] not in key_thresholds:
            continue
        thresh = r["threshold"]
        rf_model = r["model"]
        tfidf_model = r["tfidf"]
        n_struct = len(sfeat_cols)
        X_va = r["X_va"]
        y_va = r["y_va"]

        importances = rf_model.feature_importances_[n_struct:]
        tfidf_names = list(tfidf_model.get_feature_names_out())

        if hasattr(X_va, "toarray"):
            text_feat = X_va.toarray()[:, n_struct:]
        else:
            text_feat = X_va[:, n_struct:]

        at_risk_mean = text_feat[y_va == 1].mean(axis=0) if (y_va == 1).sum() > 0 else np.zeros(len(tfidf_names))
        safe_mean = text_feat[y_va == 0].mean(axis=0)
        direction = at_risk_mean - safe_mean

        wdf = pd.DataFrame({"word": tfidf_names, "importance": importances, "direction": direction})
        red = wdf[wdf["direction"] > 0].nlargest(top_n, "importance")
        threshold_words[thresh] = red["word"].tolist()

    # Plot: heatmap of word presence across thresholds
    all_words = []
    for words in threshold_words.values():
        all_words.extend(words)
    unique_words = list(dict.fromkeys(all_words))  # preserve order, deduplicate

    presence = pd.DataFrame(0, index=unique_words, columns=[f"< {t}" for t in key_thresholds])
    for thresh, words in threshold_words.items():
        for rank, word in enumerate(words):
            presence.loc[word, f"< {thresh}"] = top_n - rank  # higher = more important

    fig, ax = plt.subplots(figsize=(8, max(6, len(unique_words) * 0.3)))
    sns.heatmap(presence, annot=True, fmt="d", cmap="Reds", ax=ax, linewidths=0.5,
                cbar_kws={"label": "Rank (higher = more important)"})
    ax.set_title(f"Top {top_n} red-flag words by threshold (0 = not in top {top_n})")
    ax.set_xlabel("Score threshold")
    ax.set_ylabel("Review word/phrase")
    plt.tight_layout()
    plt.show()

    # Words that appear at ALL thresholds (most robust signals)
    common = set(threshold_words[key_thresholds[0]])
    for thresh in key_thresholds[1:]:
        common &= set(threshold_words[thresh])

    if common:
        print(f"\nRobust red flags (appear at ALL thresholds {key_thresholds}):")
        for w in sorted(common):
            print(f"  - {w}")
    else:
        print(f"\nNo single word appears in the top {top_n} at all thresholds.")
        print("This suggests the predictive review signal is threshold-dependent.")

    return threshold_words

threshold_words = compare_red_flags_across_thresholds(sweep, SFEAT_SWEEP, key_thresholds=[92, 95, 97])

## 4.7 User comfort level guide

This translates the threshold sweep into actionable guidance for app users.

In [ ]:
def user_comfort_guide(sweep_results):
    """Map thresholds to user-facing comfort levels with model quality stats.\n    \n    Presents the tradeoff: stricter thresholds catch fewer problems but with\n    higher confidence; lenient thresholds cast a wider net but with more false alarms.\n    \"\"\"\n    comfort_levels = {\n        98: (\"Very Strict\", \"Flags any restaurant scoring below 98. High false-alarm rate\\n\"\n             \"but catches almost all hygiene issues. For users with zero tolerance.\"),\n        97: (\"Strict\", \"Flags below 97. Good for immunocompromised individuals or\\n\"\n             \"parents with small children. Catches ~33% of restaurants.\"),\n        95: (\"Moderate\", \"Flags below 95. Balanced tradeoff — catches meaningful\\n\"\n             \"hygiene gaps without too many false alarms. Recommended default.\"),\n        93: (\"Relaxed\", \"Flags below 93. Only catches notable problems.\\n\"\n             \"Good for users who want to avoid only the worst offenders.\"),\n        90: (\"Lenient\", \"Flags below 90 (official B-grade threshold). Only catches\\n\"\n             \"restaurants that would actually fail inspection. Very few false alarms.\"),\n    }\n\n    print(\"USER COMFORT LEVEL GUIDE\")\n    print(\"=\" * 80)\n    print(f\"{'Level':15s} {'Threshold':10s} {'Sensitivity':12s} {'Specificity':12s} \"\n          f\"{'Precision':10s} {'F1':6s} {'AUC':6s}\")\n    print(\"-\" * 80)\n\n    for r in sorted(sweep_results, key=lambda x: x[\"threshold\"], reverse=True):\n        t = r[\"threshold\"]\n        if t not in comfort_levels:\n            continue\n        name, desc = comfort_levels[t]\n        print(f\"{name:15s} < {t:<8d} {r['sensitivity']:11.3f} {r['specificity']:11.3f} \"\n              f\"{r['precision']:9.3f} {r['macro_f1']:5.3f} {r['auc']:5.3f}\")\n\n    print(\"\\n\" + \"=\" * 80)\n    print(\"\\nHOW TO READ THIS:\")\n    print(\"  Sensitivity = % of truly at-risk restaurants we catch\")\n    print(\"  Specificity = % of safe restaurants we correctly clear\")\n    print(\"  Precision   = when we flag a restaurant, how often are we right?\")\n    print(\"\\nTRADEOFF:\")\n    print(\"  Stricter threshold -> catches more problems -> more false alarms\")\n    print(\"  Lenient threshold  -> fewer false alarms   -> misses subtle issues\")\n\n    # Visual guide\n    fig, ax = plt.subplots(figsize=(10, 3))\n    for i, t in enumerate(sorted(comfort_levels.keys())):\n        name, _ = comfort_levels[t]\n        matching = [r for r in sweep_results if r[\"threshold\"] == t]\n        if not matching:\n            continue\n        r = matching[0]\n        color_val = r[\"macro_f1\"]\n        ax.barh(i, r[\"macro_f1\"], color=plt.cm.RdYlGn(color_val), edgecolor=\"white\", height=0.6)\n        ax.text(r[\"macro_f1\"] + 0.01, i, f\"F1={r['macro_f1']:.3f}\", va=\"center\", fontsize=9)\n        ax.text(0.01, i, f\"{name} (< {t})\", va=\"center\", fontsize=10, fontweight=\"bold\", color=\"white\")\n\n    ax.set_xlabel(\"Macro F1\")\n    ax.set_title(\"Model performance by user comfort level\")\n    ax.set_yticks([])\n    plt.tight_layout()\n    plt.show()\n\n\nuser_comfort_guide(sweep)

## 4.8 Experiment conclusions (EX4, EX5, EX6)

**Interpretation (EX4):**
- The 3-class (A/B/C) framing is intractable due to extreme imbalance
- Score-threshold models give us a *continuum* of sensitivity/specificity tradeoffs
- Review text (TF-IDF) adds predictive signal beyond just Google rating alone
- The red-flag words reveal what review language correlates with lower inspection scores

**Decision informed (EX5):**
- Deploy the model as a **configurable threshold tool**, not a fixed A/B/C classifier
- Let users pick their comfort level (lenient through very strict)
- The app should surface the red-flag words as "what to look for" guidance

**Recommendations (EX6):**
1. Use the threshold with the best macro-F1 as the default in the app
2. Display red-flag and green-flag review keywords to educate users
3. Future work: retrain DistilBERT at the optimal threshold for better NLP performance
4. The robust red-flag words (appearing across multiple thresholds) are the strongest
   hygiene signals and should be highlighted in the app UI